# 02 · Tratamento e base analítica

**Projeto:** Carteira de crédito e inadimplência no Sistema Financeiro Nacional
**Etapa:** tratamento dos dados brutos e geração da base para análise

**Resultado:** 36 arquivos mensais (out/2023 a set/2026) consolidados em uma base de **477 mil linhas e 21 MB** em Parquet (`data/processed/credito.parquet`), mais de 100 vezes menor que os dados brutos.

### Decisões de tratamento

| Decisão | Justificativa |
|---|---|
| Janela de 36 meses (out/2023 a set/2026) | Três anos permitem ver tendência e sazonalidade sem inflar a base |
| Processamento mês a mês | Os brutos somam cerca de 4,5 GB; cada mês é lido, agregado e descartado da memória |
| Recortes mantidos: data, UF, segmento, cliente (PF/PJ), porte, modalidade e origem | São os recortes usados nas análises e no painel |
| Recortes descartados: CNAE/ocupação, submodalidade e indexador | Multiplicam o número de linhas sem uso previsto nas análises |
| `numero_de_operacoes` excluída | O valor `-1` em contagens não informadas tornaria qualquer soma incorreta |
| Recortes como `category` e saída em Parquet | Reduzem memória e tamanho em disco, e preservam os tipos das colunas |

In [1]:
import re
import time
import zipfile
from pathlib import Path

import pandas as pd

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

INICIO = pd.Timestamp("2023-10-01")

DIMENSOES = ["data_base", "uf", "segmento", "cliente", "porte", "modalidade", "origem"]
VALORES = [
    "carteira_a_vencer",
    "vencido_de_15_ate_90_dias",
    "vencido_acima_de_90_dias",
    "carteira_vencida",
    "carteira_ativa",
    "carteira_inadimplencia",
    "ativo_problematico",
]

## 1. Seleção dos arquivos

Seleciona os CSV mensais dentro da janela de análise, identificando ano e mês pelo nome do arquivo.

In [2]:
arquivos = []
for zip_path in sorted(RAW.glob("scrdata_*.zip")):
    with zipfile.ZipFile(zip_path) as z:
        for nome in sorted(z.namelist()):
            if not nome.lower().endswith(".csv"):
                continue
            achado = re.search(r"(20\d{2})(\d{2})", nome)
            if achado:
                mes = pd.Timestamp(year=int(achado.group(1)), month=int(achado.group(2)), day=1)
                if mes < INICIO:
                    continue
            arquivos.append((zip_path, nome))

print(f"{len(arquivos)} arquivos selecionados")

36 arquivos selecionados


## 2. Leitura e agregação mensal

Cada arquivo é lido apenas com as colunas necessárias e agregado pela combinação de recortes, somando os saldos.

In [3]:
def processar_mes(zip_path, nome):
    """Lê um CSV mensal e o agrega pelos recortes de análise."""
    for encoding in ["utf-8", "latin-1"]:
        try:
            with zipfile.ZipFile(zip_path) as z, z.open(nome) as f:
                df = pd.read_csv(
                    f,
                    sep=";",
                    decimal=",",
                    encoding=encoding,
                    usecols=DIMENSOES + VALORES,
                    dtype={coluna: "category" for coluna in DIMENSOES},
                )
            break
        except UnicodeDecodeError:
            continue
    else:
        raise ValueError(f"Não foi possível ler {nome}")

    return df.groupby(DIMENSOES, observed=True, as_index=False)[VALORES].sum()


partes = []
inicio_total = time.time()
for i, (zip_path, nome) in enumerate(arquivos, start=1):
    partes.append(processar_mes(zip_path, nome))
    print(f"[{i}/{len(arquivos)}] {nome}: {len(partes[-1]):,} linhas")

base = pd.concat(partes, ignore_index=True)
print(f"\nTotal: {len(base):,} linhas em {(time.time() - inicio_total) / 60:.1f} min")

[1/36] scrdata_202310.csv: 13,417 linhas
[2/36] scrdata_202311.csv: 13,408 linhas
[3/36] scrdata_202312.csv: 13,404 linhas
[4/36] scrdata_202401.csv: 13,475 linhas
[5/36] scrdata_202402.csv: 13,400 linhas
[6/36] scrdata_202403.csv: 13,460 linhas
[7/36] scrdata_202404.csv: 13,448 linhas
[8/36] scrdata_202405.csv: 13,516 linhas
[9/36] scrdata_202406.csv: 13,217 linhas
[10/36] scrdata_202407.csv: 13,484 linhas
[11/36] scrdata_202408.csv: 13,501 linhas
[12/36] scrdata_202409.csv: 13,540 linhas
[13/36] scrdata_202410.csv: 13,531 linhas
[14/36] scrdata_202411.csv: 13,515 linhas
[15/36] scrdata_202412.csv: 13,490 linhas
[16/36] scrdata_202501.csv: 13,389 linhas
[17/36] scrdata_202502.csv: 13,328 linhas
[18/36] scrdata_202503.csv: 13,384 linhas
[19/36] scrdata_202504.csv: 13,402 linhas
[20/36] scrdata_202505.csv: 13,576 linhas
[21/36] scrdata_202506.csv: 13,608 linhas
[22/36] scrdata_202507.csv: 12,773 linhas
[23/36] scrdata_202508.csv: 12,824 linhas
[24/36] scrdata_202509.csv: 12,852 linhas
[

## 3. Padronização

Conversão da data-base, remoção de espaços nos textos e conversão dos recortes para `category`.

In [4]:
base["data_base"] = pd.to_datetime(base["data_base"].astype(str))
base = base[base["data_base"] >= INICIO].copy()

for coluna in DIMENSOES[1:]:
    base[coluna] = base[coluna].astype(str).str.strip().astype("category")

print("Período:", base["data_base"].min().date(), "a", base["data_base"].max().date())
print("Meses:", base["data_base"].nunique())
print(f"Linhas: {len(base):,}")
print(f"Memória: {base.memory_usage(deep=True).sum() / 1e6:.0f} MB")

Período: 2023-10-31 a 2026-09-30
Meses: 36
Linhas: 477,353
Memória: 33 MB


## 4. Validação

### 4.1 Domínio dos recortes

In [5]:
for coluna in DIMENSOES[1:]:
    valores = sorted(base[coluna].unique())
    print(f"\n{coluna} ({len(valores)}):")
    print("  " + " | ".join(valores))


uf (27):
  AC | AL | AM | AP | BA | CE | DF | ES | GO | MA | MG | MS | MT | PA | PB | PE | PI | PR | RJ | RN | RO | RR | RS | SC | SE | SP | TO

segmento (8):
  Arrendamento | Banco | Cooperativa | Desenvolvimento/Fomento | Financeira | Fintech | Instituição de pagamento | Outros

cliente (2):
  PF | PJ

porte (13):
  Acima de 20 salários mínimos | Até 1 salário mínimo | Grande | Indisponível | Mais de 1 a 2 salários mínimos | Mais de 10 a 20 salários mínimos | Mais de 2 a 3 salários mínimos | Mais de 3 a 5 salários mínimos | Mais de 5 a 10 salários mínimos | Micro | Médio | Pequeno | Sem rendimento

modalidade (13):
  Adiantamentos a depositantes | Direitos creditórios descontados | Empréstimos | Financiamentos | Financiamentos com interveniência | Financiamentos de infraestrutura e desenvolvimento | Financiamentos de títulos e valores mobiliários | Financiamentos imobiliários | Financiamentos rurais  (ex-financiamentos rurais e agroindustriais) | Financiamentos à exportação | Financ

Observações:
- 27 UFs e 8 segmentos de instituição, com **cooperativas** identificadas separadamente.
- `porte` combina faixas de renda (PF) e porte de empresa (PJ), além da categoria "Indisponível". Nas análises, é sempre usado junto com o recorte PF/PJ.

### 4.2 Definição de inadimplência

Pela [metodologia do BCB](https://www.bcb.gov.br/content/estabilidadefinanceira/scr/scr.data/scr_data_metodologia.pdf), a inadimplência é a carteira das operações com alguma parcela em atraso acima de 90 dias dividida pela carteira ativa. A tabela compara `carteira_inadimplencia` com `vencido_acima_de_90_dias`, que considera apenas as parcelas vencidas.

In [6]:
resumo = base.groupby("data_base")[["carteira_ativa", "carteira_inadimplencia", "vencido_acima_de_90_dias"]].sum()
resumo["carteira_ativa_R$_tri"] = resumo["carteira_ativa"] / 1e12
resumo["inad_oficial_%"] = resumo["carteira_inadimplencia"] / resumo["carteira_ativa"] * 100
resumo["venc90_%"] = resumo["vencido_acima_de_90_dias"] / resumo["carteira_ativa"] * 100
resumo[["carteira_ativa_R$_tri", "inad_oficial_%", "venc90_%"]].round(2).tail(12)

,carteira_ativa_R$_tri,inad_oficial_%,venc90_%
data_base,,,
2025-10-31,7.22,4.17,2.23
2025-11-30,7.30,4.19,2.26
2025-12-31,7.44,4.10,2.23
2026-01-31,7.43,4.43,2.39
2026-02-28,7.45,4.65,2.48
2026-03-31,7.54,4.53,2.40
2026-04-30,7.56,4.75,2.55
2026-05-31,7.59,4.84,2.56
2026-06-30,7.64,4.62,2.43


**Conclusão:** `carteira_inadimplencia` corresponde à definição oficial. Ela considera o saldo total das operações com atraso acima de 90 dias e fica próxima do dobro da medida que considera só as parcelas vencidas (5,00% contra 2,63% em set/2026). É a medida adotada no projeto.

**Primeira leitura:**  a inadimplência subiu de 4,00% para 5,00% entre set/2025 e set/2026 (+1,00 p.p.). Em três anos (out/2023 a set/2026), passou de 3,39% para 5,00%, enquanto a carteira ativa cresceu 34% (de R$ 5,81 tri para R$ 7,76 tri). A distribuição desse aumento por UF, modalidade e tipo de cliente é investigada nas próximas etapas.

## 5. Exportação

In [7]:
destino = PROCESSED / "credito.parquet"
base.to_parquet(destino, index=False)

teste = pd.read_parquet(destino)
assert len(teste) == len(base), "Número de linhas divergente após a gravação"
print(f"Salvo em {destino}: {teste.shape[0]:,} linhas, {teste.shape[1]} colunas, {destino.stat().st_size / 1e6:.1f} MB")

Salvo em ..\data\processed\credito.parquet: 477,353 linhas, 14 colunas, 20.9 MB


**Próxima etapa:** carga da base no BigQuery e modelagem em estrela (tabela fato e dimensões).